# Environment Setup

This notebook demonstrates how to genotype recurrent CNVs. The required input files are:
1. QCed CNV calls for your dataset (here `yourData.CNVcalls.txt`), one row per CNV call, with chromosome, start, end, CNV type (DEL/DUP), and sample ID columns.
2. A sample manifest for your dataset (here `yourData_manifest.txt`): tab separated with header and one row per sample.

The required tools are:
1. python (with pandas installed)
2. bedtools (code to install static binary below)

In [ ]:
%%bash

wget -nv -O bedtools https://github.com/arq5x/bedtools2/releases/download/v2.28.0/bedtools
chmod a+x bedtools

Next, set cohort name and date for saving output files. Replace `yourData` with a short name for your dataset; it is used as the prefix for all output files.

In [ ]:
import datetime
import os

# Set variables
COHORT = "yourData"
TODAY = datetime.date.today().strftime("%Y-%m-%d")

# Optionally, export to environment variables for shell cells
os.environ["COHORT"] = COHORT
os.environ["TODAY"] = TODAY


# Read in and format locus definitions

The locus definitions are defined in this public spreadsheet: https://docs.google.com/spreadsheets/d/1v6bzdeFYorIJsjuBDSTdMDHQVU47IrLI/edit?pli=1&gid=1816231862#gid=1816231862

These blocks of code read in two sets of locus definitions:
1. Loci that are genotyped with 50% reciprocal overlap
2. Loci that are genotyped with 50% one-way overlap

Regions that have multiple adjacent or nested breakpoints are genotyped with one-way overlap

In [ ]:
import pandas as pd

sheet_id = "1v6bzdeFYorIJsjuBDSTdMDHQVU47IrLI"  # your sheet ID
gid = "1816231862"  # sheet tab ID
url_reciprocal = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=tsv&gid={gid}'
df_reciprocal = pd.read_csv(url_reciprocal, sep="\t")

bed_reciprocal = df_reciprocal.loc[:,['CHR', 'START', 'END', 'Locus name']]
bed_reciprocal.to_csv('recurrentCNV_hg38.reciprocal.bed', index=False, sep='\t', header=False)
!head recurrentCNV_hg38.reciprocal.bed

In [ ]:
sheet_id = "1v6bzdeFYorIJsjuBDSTdMDHQVU47IrLI"  # your sheet ID
gid = "1406215423"  # sheet tab ID
url_oneway = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=tsv&gid={gid}'

df_oneway = pd.read_csv(url_oneway, sep="\t")

bed_oneway = df_oneway.loc[:,['CHR', 'START', 'END', 'Locus name']]
bed_oneway.columns = ['locusCHR', 'locusSTART', 'locusEND', 'locusNAME']
bed_oneway.to_csv('recurrentCNV_hg38.oneway.bed', index=False, sep='\t', header=False)
!head recurrentCNV_hg38.oneway.bed

# Reformat CNV calls

bedtools requires the first three columns of an input .bed file to be chromosome, start position, and end position. We also need to include CNV type (DEL/DUP) and sample ID.

This next block of code extracts the relevant columns from your CNV calls file (`yourData.CNVcalls.txt`), skipping the header line, and removes the "chr" prefix from the chromosome column. The example below assumes chromosome is in column 1, start in column 7, end in column 8, CNV type in column 9, and sample ID in column 10. Change the column numbers in the `awk` command to match your file.

In [ ]:
%%bash
output="${COHORT}.CNVcalls.consensus.${TODAY}.bed"
: > "$output"  # empty file

awk 'FNR>1 { sub(/^chr/, "", $1); print $1 "\t" $7 "\t" $8 "\t" $9 "\t" $10 }' yourData.CNVcalls.txt >> "$output"


head "$output"
ls -lh "$output"

# Run bedtools with 50%  reciprocal overlap

In [ ]:
%%bash

./bedtools intersect \
      -a recurrentCNV_hg38.reciprocal.bed \
      -b ${COHORT}.CNVcalls.consensus.${TODAY}.bed\
      -header -wb -wa -f .5 -r > ${COHORT}.reciprocal.${TODAY}.bed
      
head ${COHORT}.reciprocal.${TODAY}.bed
ls -lh ${COHORT}.reciprocal.${TODAY}.bed

Read them into a pandas dataframe

In [ ]:
reciprocal_calls = pd.read_csv(f'{COHORT}.reciprocal.{TODAY}.bed', sep='\t')
header = ['locusCHROM', 'locusSTART', 'locusEND', 'locusName', 'CHROM', 'START', 'END', 'TYPE', 'sampleID']
reciprocal_calls.columns = header
reciprocal_calls.head()


# Run bedtools with 50% one-way overlap

In [ ]:
%%bash

./bedtools intersect \
      -a recurrentCNV_hg38.oneway.bed \
      -b ${COHORT}.CNVcalls.consensus.${TODAY}.bed\
      -header -wb -wa -f .5 > ${COHORT}.oneway.${TODAY}.bed
      
head ${COHORT}.oneway.${TODAY}.bed

Read them into a pandas dataframe

In [ ]:
oneway_calls = pd.read_csv(f'{COHORT}.oneway.{TODAY}.bed', sep='\t')
header = ['locusCHROM', 'locusSTART', 'locusEND', 'locusName', 'CHROM', 'START', 'END', 'TYPE', 'sampleID']
oneway_calls.columns = header
oneway_calls.head()

# Format calls into genotype matrix

Initialize the genotype matrix from your sample manifest (`yourData_manifest.txt`). Any table that contains one row for each sample ID works. Set the 'identifier' column to be the sample IDs that match up with the CNV calls (here the manifest column `sampleID`; change it to the name of the ID column in your manifest).


In [ ]:
sampleQC = 'yourData_manifest.txt'
genomatrix = pd.read_csv(sampleQC, sep="\t")
genomatrix['identifier'] = genomatrix['sampleID']  # column with IDs matching the CNV calls
genomatrix.head()

Next, add reciprocal calls. You might have to change line 2 if your .bed file contained values other than 'DEL' and 'DUP'. For example, you might have 'deletion' and 'duplication', or 'loss' and 'gain'.

In [ ]:
# Map CNV types to numeric codes
cn_mapping = {'DEL': 1, 'DUP': 3}
calls_df = reciprocal_calls.copy()
calls_df['CN'] = calls_df['TYPE'].map(cn_mapping)

# Get all unique loci
loci = calls_df['locusName'].unique()
loci_reciprocal = loci

# Ensure 'identifier' and 'sampleID' are string type for consistent merging
genomatrix['identifier'] = genomatrix['identifier'].astype(str)
calls_df['sampleID'] = calls_df['sampleID'].astype(str)

# Initialize columns in genomatrix_CNV with default value 2
for l in loci:
    genomatrix[l] = 2

# Pivot CNV calls to wide format (sample x locus)
pivoted = calls_df.pivot_table(index='sampleID', columns='locusName', values='CN', aggfunc='first')

# Merge CNV calls into genotype matrix using update
pivoted.index.name = 'identifier'  # match index name to merge key
genomatrix.set_index('identifier', inplace=True)
genomatrix.update(pivoted)
genomatrix.reset_index(inplace=True)

Next, add one-way calls. We add the suffix "oneway" to the locus names to indicate that these are not yet "allele specific". You might have to change line 2 if your .bed file contained values other than 'DEL' and 'DUP'. For example, you might have 'deletion' and 'duplication', or 'loss' and 'gain'.

In [ ]:
# Map CNV types to numeric codes
cn_mapping = {'DEL': 1, 'DUP': 3}
calls_df = oneway_calls.copy()
calls_df['CN'] = calls_df['TYPE'].map(cn_mapping)

# Get all unique loci
loci = calls_df['locusName'].unique()
loci_oneway = [f'{l}_oneway' for l in loci]

# Ensure 'identifier' and 'sampleID' are string type for consistent merging
genomatrix['identifier'] = genomatrix['identifier'].astype(str)
calls_df['sampleID'] = calls_df['sampleID'].astype(str)
calls_df['locusName'] = calls_df['locusName'] + '_oneway'

# Initialize columns in genomatrix_CNV with default value 2
for l in loci:
    genomatrix[f'{l}_oneway'] = 2

# Pivot CNV calls to wide format (sample x locus)
pivoted = calls_df.pivot_table(index='sampleID', columns='locusName', values='CN', aggfunc='first')

# Merge CNV calls into genotype matrix using update
pivoted.index.name = 'identifier'  # match index name to merge key
genomatrix.set_index('identifier', inplace=True)
genomatrix.update(pivoted)
genomatrix.reset_index(inplace=True)
genomatrix.head()

Finally, add columns for loci that were not in your dataset (populate with 2s)

In [ ]:
!awk -F'\t' '{print $4}' recurrentCNV_hg38.reciprocal.bed > loci_names.txt
!awk -F'\t' '{print $4 "_oneway"}' recurrentCNV_hg38.oneway.bed >> loci_names.txt
with open('loci_names.txt') as f:
    loci = f.read().split('\n')

for l in loci:
    if l not in genomatrix.columns:
        genomatrix[l] = 2

# (Optional) set calls for samples that failed QC to NA

This assumes your manifest has a sample QC column (here `QC`) with the value 'pass' for samples that passed QC. Change the column name and value to match your manifest, or skip this step if your CNV calls are already QC filtered.

In [ ]:
import numpy as np
all_loci = list(loci_oneway) + list(loci_reciprocal)
genomatrix.loc[genomatrix["QC"] != 'pass', all_loci] = np.nan

# Combine one-way calls into "allele specific" genotypes


## 1q21.1 TAR

In [ ]:
genomatrix['1q21.1_TAR_BP1_BP2'] = np.where(
    genomatrix['1q21.1_TAR_BP2_BP3_oneway'] == 2,
    genomatrix['1q21.1_TAR_BP1_BP2_oneway'],
    2
)

genomatrix['1q21.1_TAR_BP2_BP3'] = np.where(
    genomatrix['1q21.1_TAR_BP1_BP2_oneway'] == 2,
    genomatrix['1q21.1_TAR_BP2_BP3_oneway'],
    2
)

genomatrix['1q21.1_TAR_BP1_BP3'] = np.where(
    genomatrix['1q21.1_TAR_BP1_BP2_oneway'] == genomatrix['1q21.1_TAR_BP2_BP3_oneway'],
    genomatrix['1q21.1_TAR_BP2_BP3_oneway'],
    2
)

pd.crosstab(genomatrix['1q21.1_TAR_BP2_BP3'], genomatrix['1q21.1_TAR_BP1_BP2'])

## 15q11-13

This region has two separate CNV regions: BP1-3 and BP3-5. CNVs at CYFIP and CHRNA7 are common ( ~ 0.5% AF), so it is not unlikely that there are subjects who have one of these CNVs AND an independent CNV at the other end of the region. So, we genotype BP1-BP3 and BP3-BP5 independently

In [ ]:
regions_15q13 = ['15q11.2_BP1_BP2_oneway', 
               '15q11.2-13.1_BP2_BP3_oneway']
genomatrix['15q13'] = (
    genomatrix[regions_15q13]
    .fillna(0)
    .astype(int)
    .astype(str)
    .agg(''.join, axis=1)
)

## 1-2 CYFIP

conditions = [
    genomatrix['15q13'] == '12',
    genomatrix['15q13'] == '32',
    genomatrix['15q13'] == '00'
]
choices = [1, 3, None]

genomatrix['15q11.2_BP1_BP2'] = np.select(conditions, choices, default=2)

## 2-3 

conditions = [
    genomatrix['15q13'] == '21',
    genomatrix['15q13'] == '23',
    genomatrix['15q13'] == '00'
]
choices = [1, 3, None]

genomatrix['15q11.2-13.1_BP2_BP3'] = np.select(conditions, choices, default=2)

## 1-3 

conditions = [
    genomatrix['15q13'] == '11',
    genomatrix['15q13'] == '33',
    genomatrix['15q13'] == '00'
]
choices = [1, 3, None]

genomatrix['15q11.2-13.1_BP1_BP3'] = np.select(conditions, choices, default=2)

pd.crosstab(genomatrix['15q11.2_BP1_BP2'], genomatrix['15q11.2-13.1_BP2_BP3'])

In [ ]:
regions_15q35 = ['15q13.1-13.2_BP3_BP4_oneway',
               '15q13.3_BP4_BP5_exclCHRNA7_oneway',
               '15q13.3_CHRNA7_oneway']
genomatrix['15q35'] = (
    genomatrix[regions_15q35]
    .fillna(0)
    .astype(int)
    .astype(str)
    .agg(''.join, axis=1)
)

## 3-4 

conditions = [
    genomatrix['15q35'] == '122',
    genomatrix['15q35'] == '322',
    genomatrix['15q35'] == '000'
]
choices = [1, 3, None]

genomatrix['15q13.1-13.2_BP3_BP4'] = np.select(conditions, choices, default=2)

## 3-5 

conditions = [
    genomatrix['15q35'] == '111',
    genomatrix['15q35'] == '333',
    genomatrix['15q35'] == '000'
]
choices = [1, 3, None]

genomatrix['15q13.1-13.3_BP3_BP5'] = np.select(conditions, choices, default=2)

## 4-5 

conditions = [
    genomatrix['15q35'] == '211',
    genomatrix['15q35'] == '233',
    genomatrix['15q35'] == '000'
]
choices = [1, 3, None]

genomatrix['15q13.3_BP4_BP5'] = np.select(conditions, choices, default=2)

## 4-5 excl CHRNA7

conditions = [
    genomatrix['15q35'] == '212',
    genomatrix['15q35'] == '232',
    genomatrix['15q35'] == '000'
]
choices = [1, 3, None]

genomatrix['15q13.3_BP4_BP5_exclCHRNA7'] = np.select(conditions, choices, default=2)


## CHRNA7 

conditions = [
    genomatrix['15q35'] == '221',
    genomatrix['15q35'] == '223',
    genomatrix['15q35'] == '000'
]
choices = [1, 3, None]

genomatrix['15q13.3_CHRNA7'] = np.select(conditions, choices, default=2)


pd.crosstab(genomatrix['15q13.3_CHRNA7'], genomatrix['15q13.3_BP4_BP5'])

## 16p11.2

In [ ]:
regions_16p11 = ['16p11.2_BP1_BP2_oneway', 
                 '16p11.2_BP2_BP3_oneway', 
                 '16p11.2_BP3_BP4_oneway',
                 '16p11.2_BP4_BP5_oneway']
genomatrix['16p11.2'] = (
    genomatrix[regions_16p11]
    .fillna(0)
    .astype('int8')   # much smaller memory footprint
    .astype(str)
    .agg(''.join, axis=1)
)

## 1-2

conditions = [
    genomatrix['16p11.2'] == '1222',
    genomatrix['16p11.2'] == '3222',
    genomatrix['16p11.2'] == '0000'
]
choices = [1, 3, None]

genomatrix['16p11.2_BP1_BP2'] = np.select(conditions, choices, default=2)

## 1-3

conditions = [
    genomatrix['16p11.2'] == '1122',
    genomatrix['16p11.2'] == '3322',
    genomatrix['16p11.2'] == '0000'
]
choices = [1, 3, None]

genomatrix['16p11.2_BP1_BP3'] = np.select(conditions, choices, default=2)

## 2-3

conditions = [
    genomatrix['16p11.2'] == '2122',
    genomatrix['16p11.2'] == '2322',
    genomatrix['16p11.2'] == '0000'
]
choices = [1, 3, None]

genomatrix['16p11.2_BP2_BP3'] = np.select(conditions, choices, default=2)

## 3-4

conditions = [
    genomatrix['16p11.2'] == '2212',
    genomatrix['16p11.2'] == '2232',
    genomatrix['16p11.2'] == '0000'
]
choices = [1, 3, None]

genomatrix['16p11.2_BP3_BP4'] = np.select(conditions, choices, default=2)


## 4-5

conditions = [
    genomatrix['16p11.2'] == '2221',
    genomatrix['16p11.2'] == '2223',
    genomatrix['16p11.2'] == '0000'
]
choices = [1, 3, None]

genomatrix['16p11.2_BP4_BP5'] = np.select(conditions, choices, default=2)

pd.crosstab(genomatrix['16p11.2_BP4_BP5'], genomatrix['16p11.2_BP2_BP3'])

## 16p12.2

In [ ]:

genomatrix['16p12.2_BP1_BP2'] = np.where(
    genomatrix['16p12.2_BP2_BP3_oneway'] == 2,
    genomatrix['16p12.2_BP1_BP2_oneway'],
    2
)

genomatrix['16p12.2_BP2_BP3'] = np.where(
    genomatrix['16p12.2_BP1_BP2_oneway'] == 2,
    genomatrix['16p12.2_BP2_BP3_oneway'],
    2
)

genomatrix['16p12.2_BP1_BP3'] = np.where(
    genomatrix['16p12.2_BP1_BP2_oneway'] == genomatrix['16p12.2_BP2_BP3_oneway'],
    genomatrix['16p12.2_BP1_BP2_oneway'],
    2
)

pd.crosstab(genomatrix['16p12.2_BP1_BP2'], genomatrix['16p12.2_BP2_BP3'])

## 16p13.3

In [ ]:

genomatrix['16p13.11_BP1_BP2'] = np.where(
    genomatrix['16p13.11_BP2_BP3_oneway'] == 2,
    genomatrix['16p13.11_BP1_BP2_oneway'],
    2
)

genomatrix['16p13.11_BP2_BP3'] = np.where(
    genomatrix['16p13.11_BP1_BP2_oneway'] == 2,
    genomatrix['16p13.11_BP2_BP3_oneway'],
    2
)

genomatrix['16p13.11_BP1_BP3'] = np.where(
    genomatrix['16p13.11_BP1_BP2_oneway'] == genomatrix['16p13.11_BP2_BP3_oneway'],
    genomatrix['16p13.11_BP1_BP2_oneway'],
    2
)

pd.crosstab(genomatrix['16p13.11_BP1_BP2'], genomatrix['16p13.11_BP2_BP3'])

## 22q11.2

First, genotype the A-D region

In [ ]:
regions_22qAD = ['22q11.21_A_B_oneway', '22q11.21_B_C_oneway', '22q11.21_C_D_oneway']
genomatrix['22q11.21_AD'] = (
    genomatrix[regions_22qAD]
    .fillna(0)
    .astype(int)
    .astype(str)
    .agg(''.join, axis=1)
)

## AB

conditions = [
    genomatrix['22q11.21_AD'] == '122',
    genomatrix['22q11.21_AD'] == '322',
    genomatrix['22q11.21_AD'] == '000',
]
choices = [1, 3, None]

genomatrix['22q11.21_A_B'] = np.select(conditions, choices, default=2)

## BC

conditions = [
    genomatrix['22q11.21_AD'] == '212',
    genomatrix['22q11.21_AD'] == '232',
    genomatrix['22q11.21_AD'] == '000',
]
choices = [1, 3, None]

genomatrix['22q11.21_B_C'] = np.select(conditions, choices, default=2)


## CD

conditions = [
    genomatrix['22q11.21_AD'] == '221',
    genomatrix['22q11.21_AD'] == '223',
    genomatrix['22q11.21_AD'] == '000',
]
choices = [1, 3, None]

genomatrix['22q11.21_C_D'] = np.select(conditions, choices, default=2)



## AC

conditions = [
    genomatrix['22q11.21_AD'] == '112',
    genomatrix['22q11.21_AD'] == '332',
    genomatrix['22q11.21_AD'] == '000',
]
choices = [1, 3, None]

genomatrix['22q11.21_A_C'] = np.select(conditions, choices, default=2)


## BD

conditions = [
    genomatrix['22q11.21_AD'] == '211',
    genomatrix['22q11.21_AD'] == '233',
    genomatrix['22q11.21_AD'] == '000',
]
choices = [1, 3, None]

genomatrix['22q11.21_B_D'] = np.select(conditions, choices, default=2)


## AD

conditions = [
    genomatrix['22q11.21_AD'] == '111',
    genomatrix['22q11.21_AD'] == '333',
    genomatrix['22q11.21_AD'] == '000',
]
choices = [1, 3, None]

genomatrix['22q11.21_A_D'] = np.select(conditions, choices, default=2)


## note, there were some samples with 323... this is likely due to a fragmented call

pd.crosstab(genomatrix['22q11.21_A_D'], genomatrix['22q11.21_B_D'])


Next, genotype the D-H region

In [ ]:
regions_22qDH = ['22q11.21_D_E_oneway','22q11.21_E_F_oneway', 
                 '22q11.21_F_G_oneway', '22q11.21_G_H_oneway']

genomatrix['22q11.21_DH'] = (genomatrix[regions_22qDH]
    .fillna(0)
    .astype(int)
    .astype(str)
    .agg(''.join, axis=1)
                           )

## DE

conditions = [
    genomatrix['22q11.21_DH'] == '1222',
    genomatrix['22q11.21_DH'] == '3222',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]

genomatrix['22q11.21_D_E'] = np.select(conditions, choices, default=2)


## EF

conditions = [
    genomatrix['22q11.21_DH'] == '2122',
    genomatrix['22q11.21_DH'] == '2322',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]

genomatrix['22q11.21_E_F'] = np.select(conditions, choices, default=2)


## FG

conditions = [
    genomatrix['22q11.21_DH'] == '2212',
    genomatrix['22q11.21_DH'] == '2232',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]


genomatrix['22q11.21_F_G'] = np.select(conditions, choices, default=2)

## GH

conditions = [
    genomatrix['22q11.21_DH'] == '2221',
    genomatrix['22q11.21_DH'] == '2223',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]

genomatrix['22q11.21_G_H'] = np.select(conditions, choices, default=2)

## FH

conditions = [
    genomatrix['22q11.21_DH'] == '2211',
    genomatrix['22q11.21_DH'] == '2233',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]
genomatrix['22q11.21_F_H'] = np.select(conditions, choices, default=2)

## EH

conditions = [
    genomatrix['22q11.21_DH'] == '2111',
    genomatrix['22q11.21_DH'] == '2333',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]
genomatrix['22q11.21_E_H'] = np.select(conditions, choices, default=2)

## DH

conditions = [
    genomatrix['22q11.21_DH'] == '1111',
    genomatrix['22q11.21_DH'] == '3333',
    genomatrix['22q11.21_DH'] == '0000'
]
choices = [1, 3, None]
genomatrix['22q11.21_D_H'] = np.select(conditions, choices, default=2)


pd.crosstab(genomatrix['22q11.21_D_E'], genomatrix['22q11.21_F_H'])

# View table of counts for each CNV

In [ ]:
allloci = [a[:-7] if a[-6:] == 'oneway' else a for a in all_loci] + list(genomatrix.columns)[-40:]
allloci = list(set(allloci))
table = []

for l in allloci:
    row = {"locus": l}
    if 1 in genomatrix[l].value_counts():
        row['n_DELs'] = genomatrix[l].value_counts()[1]
    else:
        row['n_DELs'] = 0
      
    if 3 in genomatrix[l].value_counts():
        row['n_DUPs'] = genomatrix[l].value_counts()[3]
    else:
        row['n_DUPs'] = 0
    table.append(row)
    
counts = pd.DataFrame(table).sort_values(by='n_DUPs', ascending=False)
pd.set_option('display.max_rows', None)
counts

# Write output

In [ ]:

genomatrix.to_csv(f'genomatrix.recurrentCNV.{TODAY}.csv')